# Évaluation approfondie de la qualité des données — Online Retail II

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.load_data import charger_ventes_en_ligne
from src.clean_transactions import detecter_doublons

donnees = charger_ventes_en_ligne()
donnees.shape

(1067371, 8)

## Liste des pays uniques

Objectif : repérer les valeurs incohérentes ou à normaliser (ex. "EIRE", "RSA", "USA", "Unspecified").

In [2]:
sorted(donnees["country"].dropna().unique())

['Australia',
 'Austria',
 'Bahrain',
 'Belgium',
 'Bermuda',
 'Brazil',
 'Canada',
 'Channel Islands',
 'Cyprus',
 'Czech Republic',
 'Denmark',
 'EIRE',
 'European Community',
 'Finland',
 'France',
 'Germany',
 'Greece',
 'Hong Kong',
 'Iceland',
 'Israel',
 'Italy',
 'Japan',
 'Korea',
 'Lebanon',
 'Lithuania',
 'Malta',
 'Netherlands',
 'Nigeria',
 'Norway',
 'Poland',
 'Portugal',
 'RSA',
 'Saudi Arabia',
 'Singapore',
 'Spain',
 'Sweden',
 'Switzerland',
 'Thailand',
 'USA',
 'United Arab Emirates',
 'United Kingdom',
 'Unspecified',
 'West Indies']

## Descriptions incohérentes pour un même StockCode

Un même produit ne devrait avoir qu'une seule description. On cherche les codes produits
qui ont plusieurs descriptions distinctes (variations de casse, espaces, libellés différents).

In [3]:
descriptions_par_produit = (
    donnees.dropna(subset=["description"])
    .groupby("stock_code")["description"]
    .nunique()
)

produits_incoherents = descriptions_par_produit[descriptions_par_produit > 1]
print("Produits avec plusieurs descriptions :", produits_incoherents.shape[0])
produits_incoherents.sort_values(ascending=False).head(10)

Produits avec plusieurs descriptions : 1232


stock_code
20713     9
21181     7
22423     7
22734     7
23084     7
21830     6
22719     6
47566B    6
85175     6
22501     5
Name: description, dtype: int64

In [4]:
exemple_stock_code = produits_incoherents.sort_values(ascending=False).index[0]
donnees.loc[
    donnees["stock_code"] == exemple_stock_code, "description"
].value_counts()

description
JUMBO BAG OWLS                  1372
missing                            1
wrongly marked. 23343 in box       1
wrongly coded-23343                1
found                              1
Found                              1
wrongly marked 23343               1
Marked as 23343                    1
wrongly coded 23343                1
Name: count, dtype: int64

## Distribution des quantités

On regarde les extrêmes avant de décider d'un seuil de valeur extrême.

In [5]:
donnees["quantity"].describe(percentiles=[0.01, 0.05, 0.95, 0.99, 0.999])

count    1.067371e+06
mean     9.938898e+00
std      1.727058e+02
min     -8.099500e+04
1%      -3.000000e+00
5%       1.000000e+00
95%      3.000000e+01
99%      1.000000e+02
99.9%    5.000000e+02
max      8.099500e+04
Name: quantity, dtype: float64

## Distribution des prix unitaires

In [6]:
donnees["unit_price"].describe(percentiles=[0.01, 0.05, 0.95, 0.99, 0.999])

count    1.067371e+06
mean     4.649388e+00
std      1.235531e+02
min     -5.359436e+04
1%       2.100000e-01
5%       4.200000e-01
95%      9.950000e+00
99%      1.800000e+01
99.9%    2.170160e+02
max      3.897000e+04
Name: unit_price, dtype: float64

## Doublons : exacts vs potentiels

Un doublon exact est une ligne strictement identique à une autre. Un doublon potentiel
partage les mêmes colonnes clés mais peut correspondre à deux saisies légitimes du même
produit dans une même facture — il ne faut donc pas le supprimer automatiquement.

In [7]:
detecter_doublons(donnees)

{'doublons_exacts': 34335, 'doublons_potentiels': 67246}

## Résumé des factures annulées

In [8]:
factures_annulees = donnees["invoice_no"].astype(str).str.startswith("C")
print("Lignes annulées :", factures_annulees.sum())
print("Factures annulées uniques :", donnees.loc[factures_annulees, "invoice_no"].nunique())

Lignes annulées : 19494
Factures annulées uniques : 8292


## Conclusion

Constats à reprendre dans `docs/data_quality_report.md` et à traiter dans
`03_transaction_cleaning.ipynb` :

- des pays nécessitent une normalisation (EIRE, RSA, USA, Unspecified)
- certains StockCode ont plusieurs descriptions : on ne fusionne pas automatiquement,
  on garde la description la plus fréquente comme référence
- des quantités et prix extrêmes existent mais restent plausibles (clientèle de grossistes)
- des doublons exacts et potentiels existent : les exacts seront supprimés, les potentiels conservés
  et analysés au cas par cas plus tard si nécessaire